In [ ]:
# --- Importaciones ---------------------------------------------------
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Anotamos las versiones para reproducibilidad
import sklearn
print(sklearn.__version__)  # anota la versión

# --- Carga del Titanic desde seaborn ---------------------------------
import seaborn as sns

try:
    df = sns.load_dataset('titanic')   # requiere internet en Colab
except Exception:
    url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    df = pd.read_csv(url)

print(df.shape)

In [ ]:
# --- Revisar valores faltantes --------------------------------------
print(df.isnull().sum())
# age         177   <- necesita imputación
# embarked      2   <- necesita imputación
# deck         688  <- demasiados nulos, la descartamos

# --- Selección de columnas útiles -----------------------------------
cols_numericas  = ['age', 'fare', 'sibsp', 'parch']
cols_categoricas = ['sex', 'embarked', 'class']
objetivo = 'survived'

# Creamos X e y con solo las columnas seleccionadas
X = df[cols_numericas + cols_categoricas]
y = df[objetivo]

print(X.shape)

In [ ]:
# --- Train / test split ANTES de cualquier transformación -----------
semilla = 42  # constante fija para reproducibilidad

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=semilla, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# --- Transformaciones para columnas numéricas -----------------------
pipeline_numerico = Pipeline(steps=[
    ('imputar', SimpleImputer(strategy='median')),
    ('escalar', StandardScaler()),
])

In [ ]:
# --- Transformaciones para columnas categóricas ---------------------
pipeline_categorico = Pipeline(steps=[
    ('imputar', SimpleImputer(strategy='most_frequent')),
    ('codificar', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

In [ ]:
# --- ColumnTransformer: aplica cada pipeline a sus columnas --------
preprocesador = ColumnTransformer(transformers=[
    ('num', pipeline_numerico,  cols_numericas),
    ('cat', pipeline_categorico, cols_categoricas),
])

In [ ]:
# --- Inspeccionar qué produce el ColumnTransformer ------------------
# Ajustamos solo el preprocesador (sin el modelo) para poder inspeccionarlo
preprocesador.fit(X_train)

X_train_prep = preprocesador.transform(X_train)
print(f'Shape original:      {X_train.shape}')
print(f'Shape preprocesado:  {X_train_prep.shape}')
# Shape original:      (712, 7)
# Shape preprocesado:  (712, 12)

# Recuperar los nombres de las columnas resultantes
nombres = preprocesador.get_feature_names_out()
print(nombres)

In [ ]:
# --- Pipeline final: preprocesador + modelo -------------------------
pipeline = Pipeline(steps=[
    ('preprocesar', preprocesador),
    ('modelo', LogisticRegression(max_iter=1000, random_state=42)),
])

In [ ]:
# --- Entrenar el pipeline completo ----------------------------------
pipeline.fit(X_train, y_train)

# --- Evaluación sobre el conjunto de prueba -------------------------
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy en test: {acc:.4f}')

In [ ]:
# --- cross_val_score funciona directamente con el pipeline ----------
scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')

print(f'Accuracy por fold: {scores.round(4)}')
print(f'Media: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
# cross_val_score solo evalúa, no entrena el pipeline original
# Por eso necesitamos fit separado para usar el modelo después:
pipeline.fit(X_train, y_train)

In [ ]:
# --- Guardar el pipeline entrenado ----------------------------------
from pathlib import Path

ruta_modelo = Path('pipeline_titanic.pkl')

try:
    joblib.dump(pipeline, ruta_modelo)
    print(f'Pipeline guardado en: {ruta_modelo}')
except Exception as e:
    print(f'Error al guardar: {e}')

# --- Recargar y predecir -----------------------------------------
try:
    pipeline_cargado = joblib.load(ruta_modelo)
    y_pred_v2 = pipeline_cargado.predict(X_test)
    print(f'Predicciones correctas: {(y_pred_v2 == y_pred).all()}')
    # Predicciones correctas: True
except Exception as e:
    print(f'Error al cargar: {e}')

In [ ]:
# --- ENFOQUE INCORRECTO: escalar antes del split (data leakage) ----
from sklearn.preprocessing import StandardScaler
scaler_malo = StandardScaler()
# Ajuste sobre TODO X (incluye información de test) ← ERROR
X_num = df[cols_numericas].fillna(df[cols_numericas].median())
X_num_escalado = scaler_malo.fit_transform(X_num)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_num_escalado, y, test_size=0.2, random_state=semilla, stratify=y
)
modelo_malo = LogisticRegression(max_iter=1000, random_state=42)
modelo_malo.fit(X_tr, y_tr)
acc_malo = accuracy_score(y_te, modelo_malo.predict(X_te))

print(f'Accuracy con leakage (solo numéricas): {acc_malo:.4f}')
print(f'Accuracy con pipeline correcto:         {acc:.4f}')